# 22.4 Streamlit 仪表板:让模型"可交互" / Streamlit Dashboards: making models interactive

**中文**:22.3 的 FastAPI 把模型变成了给**程序**调用的 API。但很多时候,你需要给**人**(产品经理、业务方、非技术同事、甚至你自己调试)一个能**点点拖拖、立刻看到结果**的界面:一个模型演示、一个"what-if"分析器、一个数据探索仪表板。传统做法要写前端(HTML/JS/React),数据科学家望而却步。**Streamlit** 的革命性在于:*用纯 Python 脚本,几行就能生成一个交互式网页应用*——`st.slider()` 出来一个滑块、`st.line_chart()` 出来一张图,不用碰一行前端代码。它是数据科学家做**快速演示、内部工具、模型探索器**的首选。本节从零复现 Streamlit 最核心的"**反应式执行模型**"(理解它才能写对 Streamlit),用 matplotlib 模拟一个交互式仪表板长什么样,并给出真实 Streamlit 代码。
**English**: 22.3's FastAPI turned the model into an API for **programs** to call. But often you need to give **people** (product managers, business stakeholders, non-technical colleagues, even yourself for debugging) an interface to **click, drag, and instantly see results**: a model demo, a "what-if" analyzer, a data-exploration dashboard. Traditionally this meant writing a frontend (HTML/JS/React), which deters data scientists. **Streamlit**'s revolution: *with a pure Python script, a few lines generate an interactive web app* — `st.slider()` gives a slider, `st.line_chart()` gives a chart, without touching any frontend code. It's the data scientist's top choice for **quick demos, internal tools, model explorers**. This section reproduces Streamlit's core "**reactive execution model**" from scratch (understanding it is essential to writing Streamlit correctly), simulates what an interactive dashboard looks like with matplotlib, and gives real Streamlit code.

---

**中文**:**Streamlit 最反直觉、也最关键的概念:反应式重跑(reactive rerun)**。
**English**: **Streamlit's most counterintuitive and crucial concept: reactive rerun.**
- **中文**:**每次用户和任何控件交互(拖滑块、点按钮、填输入框),整个脚本会从头到尾重新运行一遍**。不是"回调函数"那种局部更新,而是**整个 .py 从上到下再跑一次**,控件返回它们的当前值,页面据此重绘。
  **Every time the user interacts with any widget (drag a slider, click a button, fill an input), the entire script reruns top to bottom.** Not a "callback" local update, but **the whole .py runs again from top**, widgets return their current values, and the page redraws accordingly.
- **中文**:这个模型让代码极其简单(就是普通的顺序 Python 脚本,没有回调地狱),但也带来一个坑:**每次交互都重跑意味着"加载模型""读大文件"这种昂贵操作会被反复执行**。解药是 **`@st.cache_data` / `@st.cache_resource`**:把昂贵结果缓存起来,重跑时直接复用,不重算。**理解"重跑 + 缓存"是写好 Streamlit 的关键**(和 22.3 "模型只加载一次"是同一个性能哲学)。
  This model makes code extremely simple (just an ordinary sequential Python script, no callback hell), but brings a trap: **rerunning on every interaction means expensive operations like "load the model" or "read a big file" run repeatedly**. The cure is **`@st.cache_data` / `@st.cache_resource`**: cache expensive results and reuse them on rerun without recomputing. **Understanding "rerun + cache" is the key to good Streamlit** (the same performance philosophy as 22.3's "load the model once").

> 💡 **面试速查 / Interview cheat-sheet（★ 数据产品/演示工具）**
> **中文**:**Streamlit**=纯 Python 脚本→交互式网页 app(无需前端), 数据科学家做**演示/内部工具/模型探索器/数据仪表板**首选。**核心执行模型**:**每次控件交互整个脚本从头重跑**(顺序脚本, 无回调)→ 简单但要用 **`@st.cache_data`(数据)/`@st.cache_resource`(模型/连接)** 缓存昂贵操作避免每次重算(=22.3 模型加载一次同哲学)。**常用**:`st.slider/selectbox/file_uploader`(输入)、`st.line_chart/pyplot/dataframe`(输出)、`st.session_state`(跨重跑保存状态)、`st.sidebar`、`st.columns`(布局)。**vs 别的**:**Gradio**(更快搭 ML 模型 demo, HuggingFace 生态)、**Dash**(Plotly, 更可定制、企业级、回调式)、**Voila**(notebook→app)、**FastAPI**(给程序的 API 而非给人的 UI)。**边界**:Streamlit 适合内部工具/原型/中小并发, **不适合高并发生产级面向公众的 App**(每交互全脚本重跑、单会话状态)。部署:Streamlit Community Cloud、容器化上云。面试金句:*"Streamlit 用纯 Python 几行生成交互式仪表板, 核心是每次控件交互整个脚本重跑的反应式模型, 所以要用 @st.cache 缓存模型/数据避免重复加载; 适合快速演示和内部工具, 高并发生产 UI 另选; 给程序用是 FastAPI, 给人看是 Streamlit/Gradio。"*
> **English**: **Streamlit** = pure Python script → interactive web app (no frontend), the data scientist's top choice for **demos/internal tools/model explorers/dashboards**. **Core execution model**: **the entire script reruns top-to-bottom on every widget interaction** (sequential script, no callbacks) → simple but requires **`@st.cache_data` (data) / `@st.cache_resource` (model/connection)** to cache expensive operations and avoid recomputing each time (same philosophy as 22.3's load-once). **Common**: `st.slider/selectbox/file_uploader` (inputs), `st.line_chart/pyplot/dataframe` (outputs), `st.session_state` (persist state across reruns), `st.sidebar`, `st.columns` (layout). **vs others**: **Gradio** (faster to spin up ML model demos, HuggingFace ecosystem), **Dash** (Plotly, more customizable, enterprise, callback-based), **Voila** (notebook→app), **FastAPI** (an API for programs, not a UI for people). **Limits**: Streamlit suits internal tools/prototypes/small-medium concurrency, **not high-concurrency production public-facing apps** (full-script rerun per interaction, per-session state). Deployment: Streamlit Community Cloud, containerize to cloud. Interview line: *"Streamlit generates interactive dashboards in a few lines of pure Python; its core is the reactive model where the whole script reruns on every widget interaction, so use @st.cache to cache the model/data and avoid reloading; it suits quick demos and internal tools, with a different choice for high-concurrency production UIs; FastAPI is for programs, Streamlit/Gradio is for people."*


In [ ]:

# ============================================================
# 从零复现 Streamlit 的"反应式重跑 + 缓存"执行模型 / reproduce the reactive rerun + cache model
# 中文:streamlit 本机没装。我们用纯 Python 模拟它的核心:每次控件值改变, 整个"脚本"函数从头重跑, 控件返回当前值;
#      模型用缓存, 重跑时不重新加载。真实 Streamlit 代码见下方。
# English: streamlit isn't installed; we simulate its core: on every widget-value change the whole "script" reruns
#      top-to-bottom, widgets return current values; the model is cached and not reloaded on rerun. Real code below.
# ============================================================
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
X,y=load_iris(return_X_y=True); NAMES=list(load_iris().target_names); FEATS=load_iris().feature_names
_clf=RandomForestClassifier(n_estimators=200, random_state=0).fit(X, y)

class MiniStreamlit:                                        # 模拟 streamlit 的运行时 / mimic streamlit's runtime
    def __init__(self): self.widgets={}; self.reruns=0; self._cache={}
    def slider(self, label, default): return self.widgets.get(label, default)   # 控件返回当前值 / widget returns current value
    def interact(self, label, value): self.widgets[label]=value                 # 用户拖动控件 / user moves a widget
    def cache_resource(self, key, build):                                        # @st.cache_resource: 只构建一次 / build once
        if key not in self._cache: self._cache[key]=build()
        return self._cache[key]

def app(st):                                               # 这就是"Streamlit 脚本", 每次交互整体重跑 / the "app script", reruns fully
    st.reruns += 1
    model=st.cache_resource("model", lambda: _clf)         # ★ 缓存:模型不会每次重跑都重新加载 / cached: not reloaded each rerun
    x=[st.slider(f, float(X[:,i].mean())) for i,f in enumerate(FEATS)]   # 四个滑块的当前值 / four sliders' current values
    pred=model.predict([x])[0]; conf=float(model.predict_proba([x])[0][pred])
    return f"预测 {NAMES[pred]} (置信 {conf:.2f})"

st=MiniStreamlit()
print("① 初次加载 / first load: ", app(st), f"  [脚本重跑次数 reruns={st.reruns}]")
st.interact(FEATS[2], 6.5)                                 # 用户把'花瓣长度'滑块拖到 6.5 → 触发整体重跑 / drag slider → full rerun
print("② 拖动花瓣长度→6.5 / drag petal length→6.5:", app(st), f"  [reruns={st.reruns}]")
st.interact(FEATS[2], 1.0)                                 # 再拖到 1.0 → 又一次整体重跑 / another full rerun
print("③ 拖动花瓣长度→1.0 / drag petal length→1.0:", app(st), f"  [reruns={st.reruns}]")
print(f"\n每次交互整个脚本重跑(共 {st.reruns} 次), 但模型只加载了一次(缓存命中):{len(st._cache)==1}")


In [ ]:

# ============================================================
# 用 matplotlib 模拟一个 Streamlit 模型探索仪表板 / simulate a Streamlit model-explorer dashboard
# 中文:真实 Streamlit 里, 拖动滑块会实时更新预测。这里用 matplotlib 画出"仪表板"的样子:
#      左=当前预测的置信度条; 右=what-if 分析(扫描花瓣长度, 看预测类别如何变化)。
# English: in real Streamlit, dragging sliders updates the prediction live. Here matplotlib renders what the "dashboard"
#      looks like: left = confidence bars for the current prediction; right = what-if (sweep petal length, see class change).
# ============================================================
import matplotlib.pyplot as plt
current=[5.8, 3.0, 4.3, 1.3]                               # 当前滑块值 / current slider values
proba=_clf.predict_proba([current])[0]
fig,ax=plt.subplots(1,2,figsize=(14,5))
# 左:当前预测的三类概率(模拟 st.bar_chart)/ current prediction probabilities
bars=ax[0].barh(NAMES, proba, color=["#4C72B0","#DD8452","#55A868"])
ax[0].set_xlim(0,1); ax[0].set_title("🌸 当前输入的预测概率 (模拟 st.bar_chart)")
for b,p in zip(bars,proba): ax[0].text(p+0.01,b.get_y()+b.get_height()/2,f"{p:.2f}",va="center",fontsize=10)
ax[0].set_xlabel(f"输入: {dict(zip(['sl','sw','pl','pw'],current))}")
# 右:what-if——扫描花瓣长度, 看预测类别怎么随之改变(交互仪表板的核心价值)/ what-if sweep
pls=np.linspace(1,7,60); preds=[_clf.predict([[5.8,3.0,pl,1.3]])[0] for pl in pls]
ax[1].plot(pls, preds, drawstyle="steps-post", lw=2, color="#C44E52")
ax[1].set_yticks([0,1,2]); ax[1].set_yticklabels(NAMES)
ax[1].set_xlabel("花瓣长度 petal length (拖动滑块)"); ax[1].set_title("🔍 What-if:花瓣长度如何改变预测")
ax[1].axvline(4.3,ls="--",color="gray"); ax[1].text(4.35,0.1,"当前值",fontsize=8,color="gray")
plt.tight_layout(); plt.savefig("/tmp/mlops04_viz.png",dpi=80); plt.show()
print("交互式仪表板的价值:业务方拖动滑块就能理解'哪个特征怎样影响预测'——比一份静态报告直观得多")


**中文**:上面是原理和模拟。下面是**真实的 Streamlit 代码**——保存成 `app.py`,运行 `streamlit run app.py` 就得到一个交互网页:
**English**: The above is principle and simulation. Below is the **real Streamlit code** — save as `app.py`, run `streamlit run app.py`, and you get an interactive web page:

```python
# app.py  —— 运行 / run:  streamlit run app.py
import streamlit as st
import joblib
from sklearn.datasets import load_iris

st.title("🌸 鸢尾花分类器 / Iris Classifier")

@st.cache_resource                          # ★ 模型只加载一次, 跨重跑复用 / model loaded once, reused across reruns
def load_model():
    return joblib.load("iris_model.joblib")
model = load_model()
names = load_iris().target_names

# 侧边栏放输入滑块 / input sliders in the sidebar
st.sidebar.header("输入特征 / Input features")
sl = st.sidebar.slider("Sepal length", 4.0, 8.0, 5.8)   # 每次拖动 → 整个脚本重跑 / each drag → full rerun
sw = st.sidebar.slider("Sepal width",  2.0, 4.5, 3.0)
pl = st.sidebar.slider("Petal length", 1.0, 7.0, 4.3)
pw = st.sidebar.slider("Petal width",  0.1, 2.5, 1.3)

# 预测并展示(每次重跑都会用当前滑块值重新计算)/ predict & display (recomputed with current slider values on each rerun)
proba = model.predict_proba([[sl, sw, pl, pw]])[0]
pred  = proba.argmax()
st.metric("预测 / Prediction", names[pred], f"{proba[pred]:.0%} 置信")
st.bar_chart({"probability": dict(zip(names, proba))})    # 概率条形图 / probability bar chart
```
**中文**:注意 `@st.cache_resource` ——没有它,每次拖动滑块都会重新 `joblib.load` 模型(因为整个脚本重跑),和 22.3 "每请求加载模型"是同一个坑。**Streamlit 的简单来自"整体重跑",而写好 Streamlit 的关键就是用缓存驯服这个重跑。**
**English**: Note `@st.cache_resource` — without it, every slider drag re-`joblib.load`s the model (because the whole script reruns), the same trap as 22.3's "load the model per request." **Streamlit's simplicity comes from "full rerun," and writing good Streamlit is about taming that rerun with caching.**


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **Streamlit 的超能力是"消灭前端",让数据科学家独立交付交互产品**:过去,数据科学家做完模型后,想给业务方一个能玩的界面,要么求前端工程师帮忙(排期、沟通成本),要么放弃(交一份静态 PPT)。Streamlit 让你用**纯 Python、几十行**就做出一个能拖滑块、看图表、上传文件的网页应用——这大幅缩短了"模型→可交互产品"的距离。对求职者这是实打实的加分项:**能快速把模型做成 demo 展示,是沟通力和落地力的体现**(面试作品集、内部汇报都用得上)。
2. **理解"反应式重跑"是写对 Streamlit 的分水岭**:Streamlit 最反直觉的一点是——**每次交互,整个脚本从头到尾重跑**。这让代码简单(顺序脚本、无回调),但也是新手所有困惑的根源:"为什么我的模型每拖一下滑块就重新加载一次?""为什么变量状态没保存?"答案都在"整体重跑"里。我们从零复现的这个模型让它一目了然:**昂贵操作(加载模型、读大文件、跑查询)必须用 `@st.cache_*` 缓存,跨状态要用 `st.session_state`**。这和 22.3 "模型只加载一次"是同一个性能哲学——**别把昂贵的一次性工作放进每次都会执行的路径**。
3. **诚实的边界:Streamlit 是"内部工具/演示"利器,不是"生产级公开 App"**。①**并发与性能**:每次交互全脚本重跑 + 每用户一个会话的状态模型,决定了它**扛不住高并发的公开流量**——它是给"几个内部用户看"设计的,不是给"百万公开用户"的。真要做面向公众的高流量数据应用,得用 Dash(更可定制、企业级)或干脆前后端分离(React + FastAPI)。②**定制性有限**:想要精细的布局、复杂的交互、品牌化 UI,Streamlit 会捉襟见肘,Dash/Plotly 或真前端更合适。③**选型**:给**程序**调用 → FastAPI(22.3);给**人**快速演示 ML 模型 → Gradio(更聚焦模型 demo)或 Streamlit;做**企业级可定制仪表板** → Dash;做**高流量公开产品** → 真前端。**结论:Streamlit 是数据科学家"快速把想法/模型变成可交互 demo"的最佳工具,理解它反应式重跑 + 缓存的执行模型就能用好;但清醒它的定位——内部工具和原型的王者,不是生产级公开应用的选择。**

**English**:
1. **Streamlit's superpower is "eliminating the frontend," letting data scientists ship interactive products independently**: previously, after building a model, giving stakeholders a playable interface meant either asking frontend engineers (scheduling, communication cost) or giving up (a static slide deck). Streamlit lets you build a web app with sliders, charts, and file upload in **pure Python, a few dozen lines** — dramatically shortening the "model → interactive product" distance. For job seekers this is a real plus: **quickly turning a model into a demo shows communication and delivery skills** (useful for portfolios and internal presentations).
2. **Understanding "reactive rerun" is the divide for writing Streamlit correctly**: Streamlit's most counterintuitive point — **the entire script reruns top-to-bottom on every interaction**. This makes code simple (sequential script, no callbacks) but is the root of all beginner confusion: "why does my model reload every slider drag?" "why isn't my variable state saved?" The answers are all in "full rerun." Our from-scratch reproduction makes it clear: **expensive operations (load model, read big files, run queries) must be cached with `@st.cache_*`, and cross-state needs `st.session_state`**. Same performance philosophy as 22.3's "load the model once" — **don't put expensive one-time work in the path that runs every time**.
3. **Honest limits: Streamlit is a champion for "internal tools/demos," not "production-grade public apps"**. ① **Concurrency and performance**: full-script rerun per interaction + a per-user session state model mean it **can't handle high-concurrency public traffic** — it's designed for "a few internal users," not "millions of public users." For high-traffic public data apps, use Dash (more customizable, enterprise) or a proper frontend/backend split (React + FastAPI). ② **Limited customization**: for fine-grained layout, complex interactions, or branded UI, Streamlit falls short; Dash/Plotly or a real frontend fits better. ③ **Tool choice**: for **programs** to call → FastAPI (22.3); for quickly demoing an ML model to **people** → Gradio (more model-demo-focused) or Streamlit; for **enterprise customizable dashboards** → Dash; for **high-traffic public products** → a real frontend. **Conclusion: Streamlit is the best tool for a data scientist to quickly turn ideas/models into interactive demos; understand its reactive-rerun + cache execution model and you'll use it well; but be clear on its positioning — king of internal tools and prototypes, not the choice for production-grade public apps.**

> 💼 **实战视角 / Practical angle**
> **中文**:Streamlit 落地:①**快速模型 demo / 内部工具**——几十行做出 what-if 分析器、数据探索器、标注工具、A/B 结果看板;②**必用缓存**:`@st.cache_resource`(模型/DB连接)、`@st.cache_data`(数据/查询结果), 否则每交互重跑重算卡到怀疑人生;③`st.session_state` 跨重跑保存状态(如多步表单、对话历史);④布局用 `st.columns/st.tabs/st.sidebar`;⑤部署:Streamlit Community Cloud(免费, 连 GitHub 自动部署)或容器化上云;⑥想更聚焦 ML demo(自动生成输入组件)用 **Gradio**, 企业级可定制用 **Dash**。**别用它**做高并发公开生产 App。面试金句:*"Streamlit 用纯 Python 几行做交互式仪表板/模型 demo, 核心是每次控件交互整个脚本重跑的反应式模型, 所以要用 @st.cache 缓存模型和数据、session_state 存状态; 它是内部工具和快速演示的利器, 但每交互全重跑决定了它不适合高并发生产 UI; 给程序用 FastAPI, 给人快速演示用 Streamlit/Gradio, 企业级仪表板用 Dash。"*
> **English**: Streamlit in practice: ① **quick model demos / internal tools** — build what-if analyzers, data explorers, labeling tools, A/B dashboards in dozens of lines; ② **must use caching**: `@st.cache_resource` (model/DB connection), `@st.cache_data` (data/query results), else every interaction reruns and recomputes painfully; ③ `st.session_state` to persist state across reruns (multi-step forms, chat history); ④ layout with `st.columns/st.tabs/st.sidebar`; ⑤ deploy via Streamlit Community Cloud (free, auto-deploy from GitHub) or containerize to cloud; ⑥ for a more ML-demo-focused tool (auto-generated input widgets) use **Gradio**, for enterprise customization use **Dash**. **Don't use it** for high-concurrency public production apps. Interview line: *"Streamlit builds interactive dashboards/model demos in a few lines of pure Python; its core is the reactive model where the whole script reruns on every widget interaction, so use @st.cache to cache the model and data and session_state for state; it's great for internal tools and quick demos, but full-rerun-per-interaction makes it unsuitable for high-concurrency production UIs; FastAPI for programs, Streamlit/Gradio for quick human demos, Dash for enterprise dashboards."*

---
### 小结 / Summary
- **中文**:Streamlit=纯 Python→交互式网页 app(无前端), 数据科学家做演示/内部工具/模型探索器首选。
- **English**: Streamlit = pure Python → interactive web app (no frontend), the data scientist's top choice for demos/internal tools/model explorers.
- **中文**:核心=反应式重跑(每次交互整个脚本重跑), 必用 @st.cache_* 缓存昂贵操作、session_state 存状态。
- **English**: Core = reactive rerun (whole script reruns per interaction); must use @st.cache_* for expensive operations and session_state for state.
- **中文**:适合内部工具/原型, 不适合高并发生产公开 App; 给程序用 FastAPI, 给人演示用 Streamlit/Gradio, 企业仪表板用 Dash。
- **English**: Suits internal tools/prototypes, not high-concurrency public apps; FastAPI for programs, Streamlit/Gradio for human demos, Dash for enterprise dashboards.
